In [1]:
import os, glob
import numpy as np
import pandas as pd
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
import cv2
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch

2026-01-16 03:05:48.615374: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-16 03:05:48.627492: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768529148.640313 2708302 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768529148.644177 2708302 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-16 03:05:48.658639: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
class SpineLandmarksDataset(Dataset):
    def __init__(self, df_long, img_size=256, sigma=5, transform=None):
        self.transform = transform

        df = df_long.copy().reset_index(drop=True)
        df["level_norm"] = df["level"].map(canonical_level_name)

        # define channel order
        self.levels = sorted(df["level_norm"].unique())
        self.level_to_idx = {lvl: i for i, lvl in enumerate(self.levels)}
        self.img_size = img_size
        self.sigma = sigma

        # group long rows by image path
        groups = {}
        for i, row in df.iterrows():
            path = row["full_path"]
            if path not in groups:
                groups[path] = []
            groups[path].append(row)

        # one sample per image
        self.samples = []
        for path, rows in groups.items():
            self.samples.append({
                "path": path,
                "rows": rows,  # list of rows for this image, one per level
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        path = sample["path"]
        rows = sample["rows"]

        # --- load & resize image as PIL ---
        img = Image.open(path).convert("L")   # "L" = single-channel grayscale
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        # --- apply transform if provided (preferred path) ---
        if self.transform is not None:
            # e.g. transforms.ToTensor() or ToTensor()+Normalize(...)
            img_t = self.transform(img)  # -> [3,H,W] float
        else:
            # fallback: manual conversion to tensor in [0,1]
            img_np = np.asarray(img, dtype=np.float32) / 255.0  # [H,W,3]
            img_np = np.transpose(img_np, (2, 0, 1))            # [3,H,W]
            img_t = torch.from_numpy(img_np)                    # float32

        H = W = self.img_size
        K = len(self.levels)
        target = np.zeros((K, H, W), dtype=np.float32)

        # --- fill heatmaps ---
        for r in rows:
            lvl = r["level_norm"]
            k = self.level_to_idx[lvl]

            rx = float(r["relative_x"])  # assumed in [0,1]
            ry = float(r["relative_y"])  # assumed in [0,1]

            cx = int(np.clip(rx, 0, 1) * (W - 1))
            cy = int(np.clip(ry, 0, 1) * (H - 1))

            target[k] = gaussian_heatmap(H, W, cx, cy, self.sigma)

        target_t = torch.from_numpy(target)  # [K,H,W]

        return img_t, target_t

In [3]:
# Dataset for dynamic augmentation

class SpineLandmarksDatasetDynamAug(Dataset):
    def __init__(self, df_long, img_size=256,
                 schedule=None,
                 normalize_mean=0.2127, normalize_std=0.2673):
        self.df = df_long.copy().reset_index(drop=True)
        self.df["level_norm"] = self.df["level"].map(canonical_level_name)

        self.levels = sorted(self.df["level_norm"].unique())
        self.level_to_idx = {lvl: i for i, lvl in enumerate(self.levels)}
        self.img_size = img_size

        # group rows by scan path (one sample = one scan)
        groups = {}
        for _, row in self.df.iterrows():
            path = row["full_path"]
            groups.setdefault(path, []).append(row)
        self.samples = [{"path": p, "rows": rows} for p, rows in groups.items()]

        # augmentation
        self.schedule = schedule
        self.current_params = schedule.params(0) if schedule is not None else None
        self.geom_aug = PairedAffineAug()
        self.int_aug = IntensityAug(fixed_blur_sigma=3.0, fixed_kernel=7)

        # normalization
        self.mean = float(normalize_mean)
        self.std = float(normalize_std)

    def set_epoch(self, epoch: int):
        if self.schedule is not None:
            self.current_params = self.schedule.params(epoch)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        path = sample["path"]
        rows = sample["rows"]

        # load grayscale
        img = Image.open(path).convert("L")
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        # to tensor [1,H,W] in [0,1]
        img_np = np.asarray(img, dtype=np.float32) / 255.0
        img_t = torch.from_numpy(img_np).unsqueeze(0)

        H = W = self.img_size
        K = len(self.levels)
        target = np.zeros((K, H, W), dtype=np.float32)

        # sigma from schedule (or fallback)
        sigma = float(self.current_params["sigma"]) if self.current_params else 5.0

        for r in rows:
            lvl = r["level_norm"]
            k = self.level_to_idx[lvl]

            rx = float(r["relative_x"])
            ry = float(r["relative_y"])

            cx = int(np.clip(rx, 0, 1) * (W - 1))
            cy = int(np.clip(ry, 0, 1) * (H - 1))

            target[k] = gaussian_heatmap(H, W, cx, cy, sigma)

        hm_t = torch.from_numpy(target)  # [K,H,W]

        # --- augment ---
        if self.current_params is not None:
            p = self.current_params

            # paired geometric
            img_t, hm_t = self.geom_aug(
                img_t, hm_t,
                rot_deg=p["rot_deg"],
                trans_frac=p["trans"],
                scale_frac=p["scale"]
            )

            # intensity (image only)
            img_t = self.int_aug(
                img_t,
                contrast_amt=p["contrast"],
                gamma_amt=p["gamma"],
                noise_std=p["noise_std"],
                blur_p=p["blur_p"], 
                lowres_size=p["lowres_size"]
            )

        # normalize AFTER intensity aug
        img_t = (img_t - self.mean) / (self.std + 1e-8)

        return img_t, hm_t


In [4]:
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

class SpineLandmarksDatasetStaticAug(Dataset):
    def __init__(
        self,
        df_long,
        img_size=256,
        sigma=5.0,
        transform=None,
        augment=False,

        # paired geometric
        rot_deg=45.0,
        translate_frac=0.10,
        scale_range=(0.85, 1.15),
        shear_deg=10.0,
        hflip_p=0.5,
        vflip_p=0.2,

        # image-only intensity
        brightness=(0.85, 1.15),
        contrast=(0.85, 1.15),
        gamma=(0.85, 1.20),
        noise_std=(0.0, 0.03),   # applied after ToTensor (inside dataset, before Normalize)
        blur_sigma=(0.0, 1.5),
        blur_kernel_max=11,

        filter_exists=True,
    ):
        self.transform = transform
        self.img_size = int(img_size)
        self.sigma = float(sigma)
        self.augment = bool(augment)

        self.rot_deg = float(rot_deg)
        self.translate_frac = float(translate_frac)
        self.scale_range = scale_range
        self.shear_deg = float(shear_deg)
        self.hflip_p = float(hflip_p)
        self.vflip_p = float(vflip_p)

        self.brightness = brightness
        self.contrast = contrast
        self.gamma = gamma
        self.noise_std = noise_std
        self.blur_sigma = blur_sigma
        self.blur_kernel_max = int(blur_kernel_max)

        df = df_long.copy().reset_index(drop=True)
        if filter_exists and ("exists" in df.columns):
            df = df[df["exists"] == True].reset_index(drop=True)

        # create level_norm
        df["level_norm"] = df["level"].map(canonical_level_name)

        self.levels = sorted(df["level_norm"].unique())
        self.level_to_idx = {lvl: i for i, lvl in enumerate(self.levels)}

        # group by image path
        groups = {}
        for _, row in df.iterrows():
            groups.setdefault(row["full_path"], []).append(row)
        self.samples = [{"path": p, "rows": rows} for p, rows in groups.items()]

    def __len__(self):
        return len(self.samples)

    def set_img_size(self, new_size: int):
        self.img_size = int(new_size)

    def set_sigma(self, new_sigma: float):
        self.sigma = float(new_sigma)

    def _load_image(self, path):
        img = Image.open(path).convert("L")
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        return img  # PIL

    def _build_targets(self, rows, H, W, sigma):
        K = len(self.levels)
        target = np.zeros((K, H, W), dtype=np.float32)

        for r in rows:
            lvl = r["level_norm"]
            k = self.level_to_idx[lvl]

            rx = float(np.clip(r["relative_x"], 0.0, 1.0))
            ry = float(np.clip(r["relative_y"], 0.0, 1.0))

            cx = int(rx * (W - 1))
            cy = int(ry * (H - 1))

            target[k] = gaussian_heatmap(H, W, cx, cy, sigma)

        return torch.from_numpy(target)  # [K,H,W]

    # --- paired geom: same random params for PIL image + tensor heatmaps
    def _sample_affine_params(self, H, W):
        angle = float(torch.empty(1).uniform_(-self.rot_deg, self.rot_deg))
        scale = float(torch.empty(1).uniform_(self.scale_range[0], self.scale_range[1]))

        max_dx = self.translate_frac * W
        max_dy = self.translate_frac * H
        tx = float(torch.empty(1).uniform_(-max_dx, max_dx))
        ty = float(torch.empty(1).uniform_(-max_dy, max_dy))

        shear_x = float(torch.empty(1).uniform_(-self.shear_deg, self.shear_deg))
        shear_y = float(torch.empty(1).uniform_(-self.shear_deg, self.shear_deg))
        return angle, scale, (tx, ty), (shear_x, shear_y)

    def _apply_paired_geom(self, img_pil, hm_t):
        # flips
        if torch.rand(()) < self.hflip_p:
            img_pil = TF.hflip(img_pil)
            hm_t = TF.hflip(hm_t)
        if torch.rand(()) < self.vflip_p:
            img_pil = TF.vflip(img_pil)
            hm_t = TF.vflip(hm_t)

        # affine
        H = W = self.img_size
        angle, scale, (tx, ty), (shx, shy) = self._sample_affine_params(H, W)
        center = [ (W - 1) / 2.0, (H - 1) / 2.0 ]

        img_pil = TF.affine(
            img_pil,
            angle=angle,
            translate=[tx, ty],
            scale=scale,
            shear=[shx, shy],
            interpolation=InterpolationMode.BILINEAR,
            fill=0,
            center=center,
        )
        hm_t = TF.affine(
            hm_t,
            angle=angle,
            translate=[tx, ty],
            scale=scale,
            shear=[shx, shy],
            interpolation=InterpolationMode.BILINEAR,
            fill=0.0,
            center=center,
        )
        return img_pil, hm_t

    # --- intensity on PIL (before ToTensor)
    def _apply_intensity_pil(self, img_pil):
        b = float(torch.empty(1).uniform_(self.brightness[0], self.brightness[1]))
        c = float(torch.empty(1).uniform_(self.contrast[0], self.contrast[1]))
        g = float(torch.empty(1).uniform_(self.gamma[0], self.gamma[1]))

        img_pil = TF.adjust_brightness(img_pil, b)
        img_pil = TF.adjust_contrast(img_pil, c)
        img_pil = TF.adjust_gamma(img_pil, gamma=g)

        sig = float(torch.empty(1).uniform_(self.blur_sigma[0], self.blur_sigma[1]))
        if sig > 1e-6:
            k = int(max(3, min(self.blur_kernel_max, 2 * int(3 * sig) + 1)))
            if k % 2 == 0:
                k += 1
            img_pil = TF.gaussian_blur(img_pil, kernel_size=[k, k], sigma=[sig, sig])

        return img_pil

    def __getitem__(self, idx):
        sample = self.samples[idx]
        path = sample["path"]
        rows = sample["rows"]

        # PIL image
        img_pil = self._load_image(path)
        H = W = self.img_size

        # heatmaps tensor
        target_t = self._build_targets(rows, H, W, sigma=self.sigma)  # [K,H,W]

        if self.augment:
            # paired geometry on PIL + heatmaps tensor
            img_pil, target_t = self._apply_paired_geom(img_pil, target_t)

            # intensity on PIL only
            img_pil = self._apply_intensity_pil(img_pil)

        # now apply your transform (ToTensor + Normalize)
        if self.transform is not None:
            img_t = self.transform(img_pil)  # ✅ ToTensor runs on PIL
        else:
            img_t = TF.to_tensor(img_pil)

        # optional: add tensor noise AFTER ToTensor and BEFORE Normalize
        # (since Normalize is in your transform, we only add noise if transform is None)
        # If you want noise even with Normalize, remove noise_std from this class
        # or split your transform into ToTensor then noise then Normalize.
        if self.noise_std is not None and self.noise_std[1] > 0 and self.transform is None:
            ns = float(torch.empty(1).uniform_(self.noise_std[0], self.noise_std[1]))
            if ns > 1e-8:
                img_t = (img_t + torch.randn_like(img_t) * ns).clamp(0.0, 1.0)

        return img_t, target_t


In [5]:
import numpy as np

def lerp(a, b, t):
    return a + (b - a) * t

class CurriculumSchedule:
    def __init__(self,
                 num_epochs,

                 # Geometry start (decays to 0)
                 rot_start=15.0,
                 trans_start=0.10,
                 scale_start=0.15,

                 # Intensity start (decays to 0)
                 contrast_start=0.30,
                 gamma_start=0.30,
                 noise_start=0.05,

                 # Blur probability schedule (strong early -> 0)
                 blur_p_start=0.95,
                 blur_p_end=0.0,
                 blur_decay_portion=0.35,

                 # Heatmap sigma schedule (easy -> precise)
                 sigma_start=9.0,
                 sigma_end=5.0):
        self.num_epochs = max(1, int(num_epochs))

        self.rot_start = float(rot_start)
        self.trans_start = float(trans_start)
        self.scale_start = float(scale_start)

        self.contrast_start = float(contrast_start)
        self.gamma_start = float(gamma_start)
        self.noise_start = float(noise_start)

        self.blur_p_start = float(blur_p_start)
        self.blur_p_end = float(blur_p_end)
        self.blur_decay_portion = float(blur_decay_portion)

        self.sigma_start = float(sigma_start)
        self.sigma_end = float(sigma_end)

    def params(self, epoch):
        t = float(np.clip(epoch / max(1, self.num_epochs - 1), 0.0, 1.0))

        # --- resolution curriculum (5 epochs 64, 10 epochs 128, 15 epochs full) ---
        if epoch <= 4:
            lowres_size = 64
        elif epoch <= 14:
            lowres_size = 128
        else:
            lowres_size = None

        # --- general augs decay to 0 by the end ---
        rot_deg = lerp(self.rot_start, 0.0, t)
        trans   = lerp(self.trans_start, 0.0, t)
        scale   = lerp(self.scale_start, 0.0, t)

        contrast = lerp(self.contrast_start, 0.0, t)
        gamma    = lerp(self.gamma_start, 0.0, t)
        noise_std = lerp(self.noise_start, 0.0, t)

        # --- targets: easy -> precise ---
        sigma = lerp(self.sigma_start, self.sigma_end, t)

        # --- blur probability: strong early -> 0 ---
        if t <= self.blur_decay_portion:
            blur_p = self.blur_p_start
        else:
            tb = (t - self.blur_decay_portion) / max(1e-8, (1.0 - self.blur_decay_portion))
            blur_p = lerp(self.blur_p_start, self.blur_p_end, tb)

        return {
            "rot_deg": float(rot_deg),
            "trans": float(trans),
            "scale": float(scale),

            "contrast": float(contrast),
            "gamma": float(gamma),
            "noise_std": float(noise_std),

            "blur_p": float(blur_p),
            "sigma": float(sigma),
            "lowres_size": lowres_size,
        }
    

In [6]:
import torch
import random
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

class PairedAffineAug:
    def __init__(self):
        pass

    def __call__(self, img_t, hm_t, rot_deg, trans_frac, scale_frac):
        # img_t: [1,H,W], hm_t: [K,H,W]
        _, H, W = img_t.shape

        angle = random.uniform(-rot_deg, rot_deg) if rot_deg > 0 else 0.0

        max_tx = trans_frac * W
        max_ty = trans_frac * H
        tx = random.uniform(-max_tx, max_tx) if trans_frac > 0 else 0.0
        ty = random.uniform(-max_ty, max_ty) if trans_frac > 0 else 0.0

        # scale factor in [1 - s, 1 + s]
        s = scale_frac
        scale = random.uniform(1.0 - s, 1.0 + s) if s > 0 else 1.0

        # Apply same affine to both
        img_t2 = TF.affine(
            img_t, angle=angle, translate=[tx, ty], scale=scale, shear=[0.0, 0.0],
            interpolation=InterpolationMode.BILINEAR
        )
        hm_t2 = TF.affine(
            hm_t, angle=angle, translate=[tx, ty], scale=scale, shear=[0.0, 0.0],
            interpolation=InterpolationMode.BILINEAR
        )
        return img_t2, hm_t2


class IntensityAug:
    def __init__(self, fixed_blur_sigma=3.0, fixed_kernel=7):
        self.fixed_blur_sigma = float(fixed_blur_sigma)
        self.fixed_kernel = int(fixed_kernel)

    def __call__(self, img_t,
                 contrast_amt, gamma_amt, noise_std,
                 blur_p,
                 lowres_size=None):
        """
        img_t: [1,H,W] in [0,1]
        lowres_size: int (e.g., 64 or 128) or None
        """
        x = img_t
        _, H, W = x.shape

        # --- resolution reduction (coarse-first) ---
        if lowres_size is not None:
            lr = int(lowres_size)
            x = TF.resize(x, [lr, lr], interpolation=InterpolationMode.BILINEAR)
            x = TF.resize(x, [H, W], interpolation=InterpolationMode.BILINEAR)

        # --- contrast ---
        if contrast_amt > 0:
            c = random.uniform(1.0 - contrast_amt, 1.0 + contrast_amt)
            x = TF.adjust_contrast(x, c)

        # --- gamma ---
        if gamma_amt > 0:
            g = random.uniform(1.0 - gamma_amt, 1.0 + gamma_amt)
            g = max(g, 0.1)
            x = TF.adjust_gamma(x, g)

        # --- noise ---
        if noise_std > 0:
            x = x + noise_std * torch.randn_like(x)

        # --- blur (fixed sigma/strength) ---
        if blur_p > 0 and random.random() < blur_p:
            k = self.fixed_kernel
            sigma = self.fixed_blur_sigma
            x = TF.gaussian_blur(x, kernel_size=[k, k], sigma=[sigma, sigma])

        return torch.clamp(x, 0.0, 1.0)
